# Tutorial 5: SSH Connection Pooling

Learn how to configure SSH ControlMaster for optimal performance when scanning remote files.

## The Problem

Without connection pooling:
- Each Snakemake rule creates new SSH connections
- Typical scan = 44+ connections (11 errors × 4 operations)
- Connection timeouts after 2 minutes
- Slow performance from repeated SSH handshakes

## The Solution: SSH ControlMaster

SSH ControlMaster shares a single connection across all processes:
- Reduces 44+ connections to 1-2 per host
- Dramatically faster connection setup
- Prevents timeouts
- Works transparently with Snakemake

## Step 1: Check Current Configuration

In [ ]:
import subprocess
import sys

# Check if as-scan check-ssh command is available
try:
    result = subprocess.run(
        ["as-scan", "check-ssh", "--help"],
        capture_output=True,
        text=True,
        timeout=5
    )
    if result.returncode == 0:
        print("✓ as-scan check-ssh command available")
        print("\nUsage:")
        print("  as-scan check-ssh [hostname]")
    else:
        print("✗ as-scan not installed or not in PATH")
except Exception as e:
    print(f"Note: {e}")
    print("Install autosubmit-scan to use check-ssh command")

## Step 2: Configure SSH ControlMaster

Add to `~/.ssh/config`:

```ssh-config
# Connection multiplexing for autosubmit-scan
Host *
    ControlMaster auto
    ControlPath ~/.ssh/control-%C
    ControlPersist 10m
```

### Configuration Breakdown

- **ControlMaster auto**: Automatically create or reuse control socket
- **ControlPath ~/.ssh/control-%C**: Socket location (%C = hash of connection details)
- **ControlPersist 10m**: Keep connection alive for 10 minutes after last use

## Step 3: Verify Configuration

In [ ]:
# Check SSH configuration for a host
import subprocess

def check_ssh_config(hostname="example.com"):
    """Check SSH configuration for ControlMaster settings."""
    try:
        result = subprocess.run(
            ["ssh", "-G", hostname],
            capture_output=True,
            text=True,
            timeout=5
        )
        
        if result.returncode == 0:
            config_lines = result.stdout.split('\n')
            control_settings = [
                line for line in config_lines 
                if 'control' in line.lower()
            ]
            
            print(f"SSH Configuration for {hostname}:")
            for line in control_settings:
                print(f"  {line}")
            
            # Check if ControlMaster is enabled
            has_master = any('controlmaster' in line.lower() and 'auto' in line.lower() 
                            for line in control_settings)
            has_path = any('controlpath' in line.lower() for line in control_settings)
            has_persist = any('controlpersist' in line.lower() for line in control_settings)
            
            if has_master and has_path and has_persist:
                print("\n✓ SSH ControlMaster properly configured")
            else:
                print("\n✗ SSH ControlMaster not fully configured")
                if not has_master:
                    print("  Missing: ControlMaster auto")
                if not has_path:
                    print("  Missing: ControlPath")
                if not has_persist:
                    print("  Missing: ControlPersist")
        else:
            print(f"Error checking SSH config: {result.stderr}")
    except Exception as e:
        print(f"Note: {e}")

# Example check (won't work without real hostname)
print("To check your SSH configuration:")
print("  Run: ssh -G [your-hostname] | grep -i control")
print("\nOr use the CLI:")
print("  as-scan check-ssh [your-hostname]")

## Step 4: Test Connection Pooling

```bash
# First connection creates control socket
ssh your-host echo "test"

# Check that control socket exists
ls -l ~/.ssh/control-*

# Subsequent connections should be instant
time ssh your-host echo "test"  # Should be < 0.1s
```

## Performance Comparison

| Scenario | Without ControlMaster | With ControlMaster |
|----------|----------------------|--------------------|
| Connections | 44 (11 errors × 4 ops) | 1-2 |
| Connection Time | ~2s × 44 = 88s | ~2s + (0.01s × 43) = 2.4s |
| Timeout Risk | High | Low |
| Total Scan Time | 90-120s | 5-10s |

## Troubleshooting

### Connection Still Timing Out?

1. **Verify SSH config**:
   ```bash
   ssh -G hostname | grep -i control
   ```

2. **Check control sockets**:
   ```bash
   ls -l ~/.ssh/control-*
   ```

3. **Increase ControlPersist**:
   ```ssh-config
   ControlPersist 30m  # Keep alive longer
   ```

4. **Enable SSH debug mode**:
   ```bash
   ssh -vvv hostname
   # Look for: "Requesting mux connect"
   ```

### Permission Issues

```bash
# Check permissions on SSH directory
chmod 700 ~/.ssh
chmod 600 ~/.ssh/config

# Control sockets should be owner-only
ls -l ~/.ssh/control-*
# Should show: srw------- (mode 0600)
```

## Security Considerations

1. **Control Sockets**: Created with mode 0600 (owner-only)
2. **Socket Location**: Stored in ~/.ssh/ (user-private)
3. **Automatic Cleanup**: Sockets removed after ControlPersist timeout
4. **Shared Systems**: Consider per-host config instead of Host *

## Advanced Configuration

### Per-Host Settings

```ssh-config
# Global default
Host *
    ControlMaster no

# Enable only for specific hosts
Host lumi mn5 *.hpc.example.com
    ControlMaster auto
    ControlPath ~/.ssh/control-%C
    ControlPersist 10m
```

### Custom Control Path

```ssh-config
# Use different path per host
Host lumi
    ControlPath ~/.ssh/control-lumi-%r@%h:%p
```

## Integration with autosubmit-scan

ControlMaster works automatically with autosubmit-scan:

```bash
# Single command - automatically uses connection pooling
as-scan scan --catalog remote_catalog.yaml --output results --cores 4
```

The workflow stages all share the same SSH connection:
1. discover_files → glob expansion
2. fingerprint_file → metadata extraction
3. match_pattern → pattern scanning
4. extract_context → context extraction

## Key Takeaways

- **Essential for Remote Scans**: ControlMaster is critical for SSH/SFTP performance
- **Easy Setup**: Add 3 lines to ~/.ssh/config
- **Huge Performance Gain**: 10-40x faster for multi-file scans
- **Transparent**: Works automatically once configured
- **Secure**: Connections protected with proper permissions

## Next Steps

- [SSH Connection Pooling Guide](../SSH_CONNECTION_POOLING.md) - Detailed technical guide
- [Tutorial 3: Remote Files](03_remote_files.ipynb) - Using SSH in catalogs
- [Performance Tuning](../how-to/performance-tuning.md) - Optimize scan performance